# Flink 风控 SQL

按事件时间统计 5 分钟窗口内的支付失败事件，达到次数或金额阈值时写入告警 Topic。

Notebook 用于理解规则、预览 SQL 和验证本地样例；真正连接 Kafka 并提交 Flink 作业时使用同目录的 `risk.sql`。

## 处理流程

- `WATERMARK` 处理可能迟到的事件。
- `TUMBLE` 将事件按 5 分钟划分窗口。
- `WHERE` 只保留支付失败事件。
- `GROUP BY` 按窗口和门店聚合。
- `HAVING` 按失败次数或金额触发告警。

In [ ]:
from datetime import datetime
from pathlib import Path
from collections import defaultdict

for candidate in [Path.cwd(), *Path.cwd().parents]:
    sql_path = candidate / "work" / "flink" / "risk.sql"
    if sql_path.is_file():
        break
else:
    sql_path = Path.cwd() / "risk.sql"

risk_sql = sql_path.read_text(encoding="utf-8")
print(f"SQL 文件：{sql_path}")
print(risk_sql)

## 本地样例

下面只用 Python 标准库模拟同一条业务规则，不启动 Kafka 或 Flink。窗口边界按 5 分钟对齐，告警条件与 `risk.sql` 保持一致。

In [ ]:
events = [
    {"event_time": "2026-07-10 10:01:00", "shop_id": "S1008", "event_type": "payment_result", "status": "PAY_FAILED", "amount": 3200.0},
    {"event_time": "2026-07-10 10:02:00", "shop_id": "S1008", "event_type": "payment_result", "status": "PAY_FAILED", "amount": 2800.0},
    {"event_time": "2026-07-10 10:04:00", "shop_id": "S1008", "event_type": "payment_result", "status": "PAY_FAILED", "amount": 2500.0},
    {"event_time": "2026-07-10 10:06:00", "shop_id": "S1008", "event_type": "order_paid", "status": "PAY_SUCCESS", "amount": 12000.0},
    {"event_time": "2026-07-10 10:07:00", "shop_id": "S2001", "event_type": "payment_result", "status": "PAY_FAILED", "amount": 11000.0},
]

def window_start(value: datetime) -> datetime:
    minute = value.minute - value.minute % 5
    return value.replace(minute=minute, second=0, microsecond=0)


failed = [
    event
    for event in events
    if event["event_type"] == "payment_result"
    and event["status"] == "PAY_FAILED"
]
groups = defaultdict(lambda: {"fail_count": 0, "fail_amount": 0.0})
for event in failed:
    event_time = datetime.fromisoformat(event["event_time"])
    key = (window_start(event_time), event["shop_id"])
    groups[key]["fail_count"] += 1
    groups[key]["fail_amount"] += event["amount"]

alerts = [
    {
        "window_start": start.isoformat(sep=" "),
        "shop_id": shop_id,
        **values,
    }
    for (start, shop_id), values in groups.items()
    if values["fail_count"] >= 3 or values["fail_amount"] >= 10000
]
alerts

## 提交到 Flink

本地样例只能验证业务规则，不能替代 Flink 的事件时间、Watermark、Kafka 连接和容错验证。确认集群配置、Topic 和权限后，从 `processing` 根目录执行：

In [ ]:
sql_client_command = "sql-client.sh -f work/flink/risk.sql"
print(sql_client_command)